# TorchRL Q-values

In this tutorial, we will start to allow our agents to learn!

* Note: this notebook was modified from the official tutorials provided by torchRL, you can check it out here: https://docs.pytorch.org/rl/stable/tutorials

### Install TorchRL

First let's install torchrl and tensordict.


In [ ]:
#Option 1: easy
#If you are running this from colab you can run:
#!pip install tensordict
#!pip install torchrl

#Option 2: harder
#if you are using your own compputer you should create a virtual environment, activate it, and then use pip to install the needed 
#conda create -n torchrl-env python=3.11 -y
#conda activate torchrl-env
#pip install torchrl tensordict torch torchvision 'gymnasium[all]'
#Note: for GPU support this might be a little different, and might vary by device!

Let's load some libraries

In [ ]:
from tensordict import TensorDict
from tensordict.nn import TensorDictModule, TensorDictSequential
from torchrl.envs import GymEnv, TransformedEnv, Compose
from torchrl.envs.transforms import RewardSum, StepCounter
from torchrl.modules import EGreedyModule
import torch
from torch import nn



# Q-values

Let's start learning about how to learn by building a familiar environment.

In [ ]:
env = GymEnv("CartPole-v1", from_pixels=True, pixels_only=False)

#add in step counter and reward sums per episode
env = TransformedEnv(env,
                     Compose(
        StepCounter(),
        RewardSum()
    )
    )

#Actions
env_actions = env.action_spec.shape[-1]
print(f"Action space: {env_actions}")

#Observations
env_obs = env.observation_spec["observation"].shape[-1]
print(f"Observation space: {env_obs}")

Then let's setup our model to go from observations to actions, then wrap this model in a module pre-built for QValueActors. This module will ensure that we feed it observations and it will return action_values. These action values will help the agent choose actions that lead to more rewards. 

We can also do the same as above but take advantage of actor modules. These modules handle common action selection patterns. Let's take a look at doing the same thing as above but just with the QvalueActor module.



In [ ]:
from torchrl.modules import QValueActor

#create a model: takes observations as inputs, and outputs actions values
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#go from observations to logits using the model
value_net = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action_value"],          
)

#use the pre-built QValueActor
qvalue_actor = QValueActor(module=value_net, spec=env.action_spec)

Now we can use this actor with the environment

In [ ]:
from torchrl.envs.utils import ExplorationType, set_exploration_type

with set_exploration_type(ExplorationType.RANDOM):
    rollout_qvalue_actor = env.rollout(max_steps=10, policy=qvalue_actor)


print(rollout_qvalue_actor)

We can see that the QValueModule added some new outputs to our tensordict:

* action_value: the current estimated value of actions

Let's take a look at the actions the agent chose.

In [ ]:
rollout_qvalue_actor["action_value"]

It also added:

* chosen_action_value: the value of the action that the agent chose

In [ ]:
rollout_qvalue_actor["chosen_action_value"]

Let's take a look at what actions the agent chose


In [ ]:
rollout_qvalue_actor["action"]


We can see that the agent is always choosing the action with the highest estimated value. Let's make it so that the agent can explore actions as well

In [ ]:
#create the e-greedy module
exploration_module = EGreedyModule(
    spec=env.action_spec, 
    eps_init=1.0,
    eps_end=0.1,
    annealing_num_steps=10000, 
    
)


#Wrap the qvalue_actor in the e-greedy module
qvalue_actor_explore = TensorDictSequential(qvalue_actor, exploration_module)

#run rollout with exploration
with set_exploration_type(ExplorationType.RANDOM):
    rollout_qvalue_actor_explore = env.rollout(max_steps=10, policy=qvalue_actor_explore)

# Let's take a look
print(rollout_qvalue_actor_explore["action"])
print(rollout_qvalue_actor_explore["action_value"])

You should now see the agent exploring actions as well as exploiting highest value actions.

### Deep q-learning (DQN)

Now that we have this working let's dig into how to use these action values to allow our agent to learn.

In TorchRL, it's possible to use dedicated loss modules which are designed with the sole purpose of optimizing the model. This approach efficiently decouples the execution of the policy from its training and allows you to design training loops that are similar to what can be found in traditional supervised learning examples.

The typical training loop therefore looks like this:

In [ ]:
# for i in range(n_data_collections):
#        data = get_next_data_batch(env, policy)
#        for j in range(n_optim):
#            loss = loss_fn(data)
#            loss.backward()
#            optim.step()

In RL, innovation typically involves the exploration of novel methods for optimizing a policy (i.e., new algorithms), rather than focusing on new architectures, as seen in other domains. Within TorchRL, these algorithms are encapsulated within loss modules. A loss module orchestrates the various components of your algorithm and yields a set of loss values that can be backpropagated through to train the corresponding components.

In this tutorial, we will take a popular off-policy algorithm as an example, DQN.



To build a loss module, the only thing one needs is a set of networks defined as TensorDictModule. Most of the time, one of these modules will be the policy. 

Let’s see what this looks like in practice: DQN requires a deterministic map from the observation space to the action space as well as a value network that predicts the value of a state-action pair. The DQN loss will attempt to find the policy parameters that output actions that maximize the value for a given state.

Let's build a policy for the DQN loss module, and then use it to alow the agent to learn a better policy.

In [ ]:

from torchrl.objectives import DQNLoss

#let's keep the observation and action shapes
#n_obs = env.observation_spec["observation"].shape[-1]
#n_act = env.action_spec.shape[-1]


#create a model: takes observations as inputs, and outputs categorical actions
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#go from observations to logits using the model
value_net = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action_value"],          
)

qvalue_actor = QValueActor(module=value_net, spec=env.action_spec)

DQN_loss = DQNLoss(qvalue_actor) #when updating use greedy (always choose best action)

Now we have a DQN loss function that will calculate how far off the agents policy was compared to what actualy happened. E.g., if the agent selected it's highest value action in a given state, did it actually get a high reward from that action? If not it should adjust it's estimation of that action in that state.

An important note here is that if we want the agent to explore using something like the e-greedy strategy, then we want to add that after building the loss function. So that when calculating the loss it assumes we always choose the best action. We'll see in more detail below why that's the case below when we build our own loss function from strach. 

For now let's add that exploration module ontop of our policy (i.e., q_value_actor).

In [ ]:
#when collecting data sometimes use random (e-greedy)
exploration_module = EGreedyModule(
    spec=env.action_spec, 
    annealing_num_steps=1000, 
    eps_init=0.95
)

#wrap the q_value_actor in a e-greedy module
qvalue_actor_explore = TensorDictSequential(qvalue_actor, exploration_module)

#let's try a rollout!
rollout = env.rollout(max_steps=100, policy=qvalue_actor_explore)

#take a look
rollout


We should see a pretty familiar rollout tensodict.

Let's look at the actions.

In [ ]:
rollout["action"]

Now that we also have a loss function we can calculate the loss from some of the observed data.

In [ ]:
loss_vals = DQN_loss(rollout)

print(loss_vals)

As you can see, the value we received from the loss isn’t a single scalar but a dictionary. This can be useful especially when the loss outputs multiple losses.

In this case we only have one loss, let's take a look at it.



In [ ]:
obs_loss = loss_vals["loss"]
print(obs_loss)


This is how far off the q_value_actor policy is from predicting the observed values of actions. We could use this loss to update the policy to make better estimates of the value of actions in states.

Let's take a look at how to use this loss to update our policies.

### Training a policy

Given all this, training a policy is not so different from what would be done in any other training loop. Because we wrap the policy as a module, the easiest way to get the list of trainable parameters is to query the parameters() method.

We’ll need an optimizer (or one optimizer per module if that is your choice).

In [ ]:
from torch.optim import Adam

#create the optimizer (let's use ADAM)
optim = Adam(DQN_loss.parameters())

#propogate the loss backwards to update the paramters of the model to better align with what the agent observed
obs_loss.backward()

Once we run the obs_loss.backwards the parameters in the *model* will be updated to produce action values that match with what the agent observed. Remember that the agent after each step sees the next environmenal state and recives a reward after performing an action in their current state. The agent can then see if the predicted value of the current action matched with the recived rewards. If it did not, this backwards propogation of the loss should nudge the model to make a better prediction. 

Over many agent-environment interactions the aim is that the agent will be better at predicting which actions in which states lead to better outcomes.

We can now use optimizers to let our agents learn! Though let's take a minute to see how exactly the loss is being calculated, as it is very important for how these agents learn.

### Manually calculate loss

Let's dig a little deeper to see if we can't better understand what the DQNLoss function is doing, and why it's doing it.

Let's see if we can calculate the loss ourselves. To do so let's:

* Take an action in the environment to collect some data
* Use outcome of the data to update our model to better predict the value of actions


Let's do a small rollout of 10 steps and use the data from the rollout to calculate the loss.

In [ ]:
#agent-environment interactions!
rollout = env.rollout(max_steps=10, policy=qvalue_actor_explore)

#collect the data
obs    = rollout["observation"]
act    = rollout["action"].argmax(dim=-1) 
next_obs = rollout["next", "observation"]
reward = rollout["next", "reward"].squeeze(-1)
done   = rollout["next", "done"].squeeze(-1).float()

Feel free to take a look at what each bit of data looks like. E.g., we can see that the rewards are just 1s, and we are using the observed rewards on the next step after the agent has taken an action. 

In [ ]:
rollout["next","reward"]

Next let's use the NN from our policy to calculate the value of actions in the current states. Here we will use the value_net

In [ ]:
#place observations into a tensordict so we can then make predictions
currrent_observations = TensorDict({"observation": obs}, batch_size=[obs.shape[0]])

#Make predictions about the value of actions
current_value = value_net(currrent_observations) #calculate value of current state, action

print(current_value["action_value"])

Next let's get the value of the action that was taken

In [ ]:
#q value of current action
q_sa = current_value["action_value"].gather(1, act.unsqueeze(-1)).squeeze(-1)

print(q_sa)


Let's now let's look at what happened next! 

We'll use the difference between what was predicted and what actually happend to update our agent.

In [ ]:
#get observations from the next step into a tensordict
next_observations = TensorDict({"observation": next_obs}, batch_size=[next_obs.shape[0]])

#get predicted value
next_value = value_net(next_obs)

print(next_value)

Let's assume that in the next state the agent will choose the best possible action (with e-greedy it might not, but we can be a little optimistic here...)

In [ ]:
next_q = next_value.max(dim=-1).values #DNQ

print(next_q)

Now we can calculate the TD target. 

In [ ]:
#Discount factor [0-1], lower values means the agent will value immediate rewards rather than rewards in the future.
gamma = 0.99

#Calculate the TD target
target = reward + gamma * (1.0 - done) * next_q

The TD target represents the agents best estimate of the overall value of the current state. It's based on the rewards the agent recieved in the next state, and the estimated value of the next state.

Once we have the target, we can see how well our model did at predicting the TD target.

* q_sa: the predicted value of the action in the current state
* target: the observed value of the action in the current state
* loss = (q_sa - target)^2

In [ ]:
#loss is the MSE between the current q_sa and the target (based on the reward observed and the discounted future q_sa)
import torch.nn.functional as F

#let's use a mean squared error loss function
loss = F.mse_loss(q_sa, target)

print(loss)

We did the rollout for 10 steps, so we can add up the loss from each step to get the total. Then we can backpropogate the loss.

 Let's look at all these steps together to get a better overall view of how the loss is caluculated and used.

In [ ]:

#agent-environment interactions!
rollout = env.rollout(max_steps=1000, policy=qvalue_actor_explore, break_when_any_done=False)

#get the required data
obs    = rollout["observation"]
act    = rollout["action"].argmax(dim=-1) 
next_obs = rollout["next", "observation"]
reward = rollout["next", "reward"].squeeze(-1)
done   = rollout["next", "done"].squeeze(-1).float()


#get current estimated value of state-action
currrent_observations = TensorDict({"observation": obs}, batch_size=[obs.shape[0]])
current_value = value_net(currrent_observations) 
q_sa = current_value["action_value"].gather(1, act.unsqueeze(-1)).squeeze(-1)

#calculate the td-target
with torch.no_grad():

    #get current estimated value of state-action in the next step
    next_observations = TensorDict({"observation": next_obs}, batch_size=[next_obs.shape[0]])
    next_value = value_net(next_obs)
    next_q = next_value.max(dim=-1).values #DNQ

    #then get the target (observed reward + discounted value of the next state)
    target = reward + gamma * (1.0 - done) * next_q

#calculate the difference between the estimated value of the current state and the td-target
loss = F.mse_loss(q_sa, target)

#let's now optimize the model used to estimate q-values
optim.zero_grad(set_to_none=True)
loss.backward() #backprop!
optim.step()


We should now know how the loss is being calculated, and how TD-targets work.

**Things to try**

* Try visualizing your cartpole agent
* Does it do better after the update? (try uncreasing the rollout size)
* Why or why not?!?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import imageio
from IPython.display import Image

#run a rollout: i.e., random policy with 200 steps
rollout = env.rollout(max_steps=200, policy=qvalue_actor, break_when_any_done=False)

#grab all the frames from the rollout
frames = rollout["pixels"].numpy()  

#save the images as a gif
imageio.mimsave("rollout.gif", frames, fps=30)

#display the gif
Image("rollout.gif")   

In [ ]:
import matplotlib.pyplot as pyplot
#get dones: when did episodes end
#done = rollout["next","?"].squeeze(-1)  # shape: [200]

#use dones to select cummulative rewards: get reward sums at the end of the episode
#episode_rewards = rollout["next", "?"][done]

#plot it
#pyplot.scatter(x=range(len(episode_rewards)), y=episode_rewards)

### Next Steps

Now that we know how to setup DQN let's put it into practice in the next tutorial!

In the meantime, feel free to:
* Check out more loss [modules](https://docs.pytorch.org/rl/stable/reference/objectives.html#ref-objectives) 
* Look at loss modules being implemented for another algorithm ([DDPG](https://docs.pytorch.org/rl/stable/tutorials/coding_ddpg.html#coding-ddpg))